# Eigendecomposition

## What's covered

- **Eigenvalues and eigenvectors** — the directions a matrix only stretches
- **Geometric meaning** — special axes that survive the transformation
- The **characteristic polynomial** `det(A - λI) = 0`
- **Diagonalization** `A = P D P^{-1}` — change basis to make the matrix diagonal
- **Symmetric matrices and the spectral theorem** — the most-loved special case
- **Matrix powers and exponentials** — diagonalization makes them trivial
- Where this appears in ML — PCA, PageRank, Hessian analysis, stability of training


## What is an eigenvector?

A matrix `A` is a function: feed it `x`, get back `Ax`. Usually `Ax` points in some new, surprising direction. But for *special* vectors, `Ax` happens to land on the same line as `x` — same direction (or exactly reversed), just scaled.

Those special vectors are the **eigenvectors** of `A`. The scaling factor is the **eigenvalue**:

$$
A \mathbf{v} = \lambda \mathbf{v}, \qquad \mathbf{v} \neq \mathbf{0}
$$

`λ` (lambda) is the eigenvalue; `v` is the eigenvector. `v = 0` is excluded — the zero vector trivially satisfies the equation and tells you nothing.

**Geometric picture.** Apply `A` to every vector on the unit circle. Most arrows get rotated *and* stretched. But along certain "magic" directions, the arrows only stretch — they keep pointing the same way (or flip). Those magic directions are the eigenvectors. The stretch factor is the eigenvalue.

**Why we care.** Eigenvectors are the **natural coordinate system** of a matrix. In that coordinate system, the matrix's action becomes simple multiplication: rescale each axis by its eigenvalue. Everything difficult about `A` — matrix powers, exponentials, stability of dynamical systems — becomes trivial in the eigenbasis.


In [ ]:
import numpy as np

# A diagonal matrix has the standard basis as its eigenvectors (the magic directions are the axes)
D = np.array([[2.0, 0.0],
              [0.0, 3.0]])
e1, e2 = np.array([1, 0.0]), np.array([0.0, 1.0])
print("D @ e1 =", D @ e1, "  = 2 * e1   <- e1 is eigenvector, eigenvalue 2")
print("D @ e2 =", D @ e2, "  = 3 * e2   <- e2 is eigenvector, eigenvalue 3")

# A general 2x2 matrix — the magic directions are no longer the axes
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
eigvals, eigvecs = np.linalg.eig(A)
print("\nA eigenvalues  =", eigvals)
print("A eigenvectors =\n", eigvecs)

# Verify Av = λv for each pair
for i in range(2):
    v = eigvecs[:, i]
    lam = eigvals[i]
    print(f"  A @ v_{i} = {A @ v},   lambda*v = {lam*v},   match? {np.allclose(A @ v, lam*v)}")


## The characteristic polynomial

How do we find eigenvalues without guessing? Rearrange the defining equation:

$$
A \mathbf{v} = \lambda \mathbf{v} \implies (A - \lambda I) \mathbf{v} = \mathbf{0}
$$

We want a *non-zero* `v` that the matrix `A - λI` sends to zero. From notebook 5, that means `A - λI` has a **non-trivial null space**, which means it is **singular**, which means:

$$
\det(A - \lambda I) = 0
$$

This is the **characteristic equation**. Expanding the determinant gives a polynomial in `λ`, called the **characteristic polynomial**. Its roots are the eigenvalues.

For our `A = [[2, 1], [1, 3]]`:

$$
\det \begin{bmatrix} 2 - \lambda & 1 \\ 1 & 3 - \lambda \end{bmatrix} = (2-\lambda)(3-\lambda) - 1 = \lambda^2 - 5\lambda + 5 = 0
$$

Quadratic formula gives `λ = (5 ± √5) / 2`, approximately `3.618` and `1.382`. Once you have `λ`, the eigenvectors come from solving `(A - λI) v = 0` — a homogeneous linear system, exactly the null-space problem from notebook 5.

Two useful sanity checks (true for any square matrix):

- The **sum of eigenvalues** equals the **trace** (sum of diagonal entries) of `A`.
- The **product of eigenvalues** equals the **determinant** of `A`.

For our `A`: trace = 5 (= 3.618 + 1.382 ✓), det = 5 (= 3.618 × 1.382 ✓).


In [ ]:
# Verify trace and determinant identities
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
eigvals, _ = np.linalg.eig(A)
print("eigenvalues =", eigvals)
print("sum         =", sum(eigvals), "  vs  trace(A) =", np.trace(A))
print("product     =", np.prod(eigvals), "  vs  det(A)   =", np.linalg.det(A))

# Manually evaluate the characteristic polynomial
# p(lambda) = lambda^2 - 5*lambda + 5
roots = np.roots([1, -5, 5])
print("\nroots of char. poly =", roots)


## Diagonalization: A = P D P^{-1}

Here is the eigendecomposition in one move. Suppose `A` is `n × n` and has `n` linearly independent eigenvectors `v_1, ..., v_n` with eigenvalues `λ_1, ..., λ_n`. Stack the eigenvectors as columns of a matrix `P`, and put the eigenvalues on the diagonal of `D`:

$$
P = [\mathbf{v}_1 \;|\; \mathbf{v}_2 \;|\; \dots \;|\; \mathbf{v}_n], \qquad D = \text{diag}(\lambda_1, \dots, \lambda_n)
$$

Then `A` factors as:

$$
A = P D P^{-1}
$$

**Read this in three steps.** Acting on a vector `x`, the right-hand side does:

1. `P^{-1} x` — re-express `x` in the eigenbasis (find its coordinates in `P`'s columns).
2. `D · (...)` — scale each coordinate by the corresponding eigenvalue.
3. `P · (...)` — translate back into the standard basis.

So `A` is *secretly* a diagonal matrix — you just have to view it from the right angle. That right angle is the eigenbasis.

**When does this work?** Whenever `A` has `n` linearly independent eigenvectors. Matrices that satisfy this are **diagonalizable**. Not every matrix is diagonalizable — *defective* matrices (like `[[1, 1], [0, 1]]`) have repeated eigenvalues without enough independent eigenvectors. SVD (next notebook) fixes this awkwardness by working for *every* matrix.


In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
eigvals, eigvecs = np.linalg.eig(A)

P = eigvecs
D = np.diag(eigvals)
P_inv = np.linalg.inv(P)

reconstructed = P @ D @ P_inv
print("P D P^-1 =\n", reconstructed)
print("matches A?", np.allclose(reconstructed, A))

# A defective matrix — not diagonalizable
defective = np.array([[1.0, 1.0],
                      [0.0, 1.0]])
ev, evec = np.linalg.eig(defective)
print("\nDefective matrix eigenvalues =", ev)
print("Eigenvectors (columns) =\n", evec, "  <- both columns are nearly parallel, not independent")
print("rank of eigenvector matrix =", np.linalg.matrix_rank(evec), "<- less than 2, so not diagonalizable")


## Symmetric matrices and the spectral theorem

The most beautiful special case. If `A` is **symmetric** (`A = A^T`), then four things are automatically true:

1. All eigenvalues are **real** (no complex numbers).
2. Eigenvectors for distinct eigenvalues are **orthogonal**.
3. Even with repeated eigenvalues, you can always find an **orthonormal** set of eigenvectors.
4. `A` is always diagonalizable.

Together these are the **spectral theorem**, and they let us write:

$$
A = Q \Lambda Q^T
$$

where `Q` is **orthogonal** (`Q^T Q = I`, columns are orthonormal eigenvectors) and `Λ` is diagonal with the eigenvalues. Compare with `A = P D P^{-1}` — for symmetric matrices, `P` is orthogonal, so `P^{-1} = P^T`. The inverse is free.

**Why ML cares so much.**

- **Covariance matrices are symmetric.** `Cov(X) = X^T X / n` is symmetric and positive semidefinite — its eigenvalues are non-negative. PCA is the spectral theorem applied to the covariance.
- **Hessians of scalar loss functions are symmetric** (by Clairaut's theorem on mixed partials). Second-order optimization (Newton's method, L-BFGS) studies eigenvalues of the Hessian to understand local curvature.
- **Kernel matrices are symmetric and PSD.** Eigendecomposition shows you their effective rank.

**Positive (semi)definite** is a sibling concept worth knowing: a symmetric matrix is **positive definite** if all eigenvalues are positive (equivalently `x^T A x > 0` for every non-zero `x`), and **positive semidefinite (PSD)** if eigenvalues are non-negative. Covariance matrices are PSD; the Hessian at a local minimum is PSD.


In [ ]:
# Symmetric matrix — use np.linalg.eigh for symmetric/Hermitian (faster, real eigenvalues)
S = np.array([[4.0, 1.0, 0.0],
              [1.0, 3.0, 1.0],
              [0.0, 1.0, 2.0]])
eigvals, Q = np.linalg.eigh(S)
print("eigenvalues =", eigvals)
print("Q orthogonal? Q^T Q ≈ I:", np.allclose(Q.T @ Q, np.eye(3)))

# Reconstruct S = Q Λ Q^T
Lambda = np.diag(eigvals)
print("\nQ Λ Q^T =\n", Q @ Lambda @ Q.T)
print("matches S?", np.allclose(Q @ Lambda @ Q.T, S))

# Positive definite? — check all eigenvalues > 0
print("\nAll eigenvalues positive?", np.all(eigvals > 0), "  → S is positive definite")


## Matrix powers and exponentials

The eigenvalue payoff: once `A = P D P^{-1}`, raising `A` to a power is *cheap*.

$$
A^k = (P D P^{-1})(P D P^{-1}) \dots (P D P^{-1}) = P D^k P^{-1}
$$

The internal `P^{-1} P = I` pairs collapse. And `D^k` is just diagonal with each entry raised to the `k`-th power. So computing `A^{1000}` becomes computing `λ_i^{1000}` — `n` scalar exponentiations.

The same trick works for the matrix exponential, defined by the power series `e^A = I + A + A²/2! + ...`:

$$
e^A = P \, e^D \, P^{-1}, \qquad e^D = \text{diag}(e^{\lambda_1}, \dots, e^{\lambda_n})
$$

**Why this matters in ML.**

- **Stability of training dynamics.** If a recurrent network's hidden-to-hidden matrix has an eigenvalue `|λ| > 1`, repeated application blows up (`λ^t → ∞`). If all `|λ| < 1`, signals vanish. The "exploding/vanishing gradient problem" in RNNs is exactly this — controlled by eigenvalues of the recurrent weight matrix.
- **PageRank.** The web-link transition matrix's top eigenvector *is* PageRank. Found by power iteration: repeatedly multiplying by `A` and renormalizing — convergence rate set by the ratio of the top two eigenvalues.
- **Markov chains.** Stationary distribution = eigenvector of the transition matrix for eigenvalue `λ = 1`.
- **Spectral clustering.** Cluster by the eigenvectors of the graph Laplacian.


In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])

# A^5 the slow way
A5_naive = np.linalg.matrix_power(A, 5)

# A^5 via diagonalization
eigvals, P = np.linalg.eig(A)
A5_eig = P @ np.diag(eigvals**5) @ np.linalg.inv(P)

print("A^5 (matrix_power) =\n", A5_naive)
print("\nA^5 (eig)          =\n", A5_eig)
print("\nmatch?", np.allclose(A5_naive, A5_eig))

# Stability check: what happens to A^k for large k?
eigmax = max(abs(eigvals))
print(f"\nlargest |eigenvalue| = {eigmax:.3f}")
print("  |λ| > 1 → A^k grows; |λ| < 1 → A^k vanishes; |λ| = 1 → A^k stays bounded")


## Where this appears in ML

Eigendecomposition is the lens through which we read every linear dynamical or geometric structure in ML.

- **PCA.** Eigenvectors of the data covariance matrix, ordered by eigenvalue. The top `k` eigenvectors span the best `k`-dimensional approximation to the data. Built carefully in the SVD notebook.
- **Hessian analysis at minima.** Eigenvalues of `∇²L` tell you curvature in each direction. Large eigenvalues = sharp, narrow valleys; small ones = wide, flat valleys. Optimizer choice (SGD vs Adam vs L-BFGS) depends heavily on this geometry.
- **Spectral norm of weight matrices.** `||W||_2` = largest singular value = largest eigenvalue of `W^T W`. Used to bound Lipschitz constants in GANs (spectral normalization), to certify robustness, and to analyze stability.
- **RNN training stability.** Recurrent weight matrices with `|λ_max| ≈ 1` keep signals from exploding or vanishing. LSTM gating is one engineered fix; orthogonal initialization (eigenvalues of magnitude 1) is another.
- **PageRank and Markov chains.** Top eigenvector of the transition matrix is the stationary distribution.
- **Spectral clustering.** Eigenvectors of the graph Laplacian reveal cluster structure when k-means in the original space fails.
- **Neural Tangent Kernel.** In the infinite-width limit, training dynamics decompose along eigenvectors of the NTK; different eigendirections converge at rates proportional to their eigenvalues.
- **Diffusion models, score-based models.** Use eigenvalue spectra of noise-perturbed data to control sampling stability.

Next notebook: **SVD and PCA** — eigendecomposition only worked for square diagonalizable matrices. SVD generalizes it to *every* matrix, and unlocks PCA, recommender systems, low-rank approximations, and the pseudoinverse all at once.
